# US Brewery Market Analysis — Final Project — SQL Queries

Group 3: Siyuan Gao, Hanqi Wang, Tianzhi Zhu

This notebook relies on Python variables generated after running the Final Analysis notebook. You need to run Final Analysis first or reintroduce the required data in the SQL notebook.

In [ ]:
import pandas as pd
import sqlite3 as sq

---
## 5. SQL Queries

We load the cleaned brewery, census, and Zillow data into a SQLite database as separate tables, then use SQL to merge them and apply window functions for regional comparison. 

### 5.1 Connect to SQLite
All SQL queries below run against a single database, making joins / group-bys / window functions reproducible.

In [ ]:
db_path = "brewery_analysis.db"
conn = sq.connect(db_path)
cursor = conn.cursor()

print(f"Connected to SQLite: {db_path}")

Connected to SQLite: brewery_analysis.db


### 5.2 Load Cleaned Tables into SQLite
Create three base tables in SQLite: `brewery_agg`, `census_agg`, `zillow_rent` by aggregating brewery to state level; selecting needed census columns; loading all three via `to_sql(replace)`.

In [ ]:
# Prepare upload DataFrames (state-level)
brewery_upload = brewery_clean.groupby(["state", "region"]).agg(
    total_breweries=("id", "count")
).reset_index()

census_upload = state_income[["state", "total_population", "weighted_average_income"]].rename(
    columns={"weighted_average_income": "weighted_avg_income"}
)

# Load to SQLite
brewery_upload.to_sql("brewery_agg", conn, if_exists="replace", index=False)
census_upload.to_sql("census_agg", conn, if_exists="replace", index=False)
zillow_state.to_sql("zillow_rent", conn, if_exists="replace", index=False)

# Quick row-count check (not counted as one of the 10 graded queries)
for t in ["brewery_agg", "census_agg", "zillow_rent"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t};", conn)["n"][0]
    print(f"{t}: {n} rows")

brewery_agg: 51 rows
census_agg: 54 rows
zillow_rent: 50 rows


### 5.3 Queries

#### 5.3.1 Query 1 (join 1): Build `state_metrics` (merged features table)
- **What:** Create a unified, state-level analysis table `state_metrics` that merges brewery, census, and rent data and computes ratio features.
- **How:** `CREATE TABLE AS SELECT ...` with joins across `brewery_agg` + `census_agg` + `zillow_rent`, plus computed columns using `CAST()` to avoid integer division.
- **Why:** A single “analysis table” makes downstream ranking, grouping, and modeling consistent and easy.

In [ ]:
cursor.executescript("""
DROP TABLE IF EXISTS state_metrics;

CREATE TABLE state_metrics AS
SELECT
    b.state,
    b.region,
    b.total_breweries,
    c.total_population,
    c.weighted_avg_income,
    r.mean_rent,

    -- Ratio features
    CAST(b.total_breweries AS FLOAT) / c.total_population * 100000.0 AS breweries_per_100k,
    c.total_population / CAST(b.total_breweries AS FLOAT)          AS people_per_brewery,
    c.weighted_avg_income / CAST(b.total_breweries AS FLOAT)        AS income_per_brewery

FROM brewery_agg b
JOIN census_agg  c ON b.state = c.state
JOIN zillow_rent r ON b.state = r.state;
""")
conn.commit()

pd.read_sql("SELECT * FROM state_metrics LIMIT 5;", conn)

,state,region,total_breweries,total_population,weighted_avg_income,mean_rent,breweries_per_100k,people_per_brewery,income_per_brewery
0,Alabama,Southeast,52,3418068,67313.745276,1148.726965,1.521327,65732.076923,1294.495101
1,Alaska,West,60,719076,93355.399874,1554.459893,8.344042,11984.600000,1555.923331
2,Arizona,Southwest,125,6900619,83511.734264,1465.823731,1.811432,55204.952000,668.093874
3,Arkansas,Southeast,45,2146827,61773.545193,1039.488558,2.096117,47707.266667,1372.745449
4,California,West,919,37513430,104267.424981,2040.855941,2.449789,40819.836779,113.457481


**SQL takeaway (Query 1):** We successfully merged brewery counts, census income/population, and Zillow rents into a single state-level table (`state_metrics`) and generated the core ratio metrics used throughout the analysis (breweries per 100k, people per brewery, income per brewery). This ensures all downstream comparisons and models rely on a consistent feature set.

#### 5.3.2 Query 2 (join 2): Data coverage audit (which states fail to match across sources)
- **What:** Identify states that appear in one source table but are missing in another (join coverage).
- **How:** Use `LEFT JOIN` from brewery to census and rent; flag missing matches via `IS NULL`.
- **Why:** If many states are missing, our merged table may be biased or incomplete.

In [ ]:
coverage_audit = pd.read_sql("""
SELECT
    b.state,
    b.region,
    CASE WHEN c.state IS NULL THEN 1 ELSE 0 END AS missing_census,
    CASE WHEN r.state IS NULL THEN 1 ELSE 0 END AS missing_rent
FROM brewery_agg b
LEFT JOIN census_agg  c ON b.state = c.state
LEFT JOIN zillow_rent r ON b.state = r.state
WHERE c.state IS NULL OR r.state IS NULL
ORDER BY missing_census DESC, missing_rent DESC, b.state;
""", conn)

coverage_audit

,state,region,missing_census,missing_rent
0,District Of Columbia,Southeast,0,1


**SQL takeaway (Query 2):** The join coverage is nearly complete. The only mismatch is District of Columbia missing Zillow rent data, so it is excluded from the fully merged dataset. This reduces the final analysis sample to 50 states with complete coverage across all sources.

#### 5.3.3 Query 3 (basic query): National baseline summary (sanity + interpretable benchmarks)
- **What:** Compute national averages for key indicators (rent, density, income, market potential).
- **How:** Aggregate `AVG()` across `state_metrics`.
- **Why:** These baselines support later subqueries (e.g., “below-average rent” or “above-average income”).

In [ ]:
national_baseline = pd.read_sql("""
SELECT
    ROUND(AVG(mean_rent), 2)            AS nat_avg_rent,
    ROUND(AVG(breweries_per_100k), 4)   AS nat_avg_density,
    ROUND(AVG(weighted_avg_income), 2) AS nat_avg_income,
    ROUND(AVG(people_per_brewery), 0)  AS nat_avg_people_per_brewery
FROM state_metrics;
""", conn)

national_baseline

,nat_avg_rent,nat_avg_density,nat_avg_income,nat_avg_people_per_brewery
0,1366.4,4.863,79768.79,33729.0


**SQL takeaway (Query 3):** National benchmarks provide interpretable reference points for “above/below average” comparisons. Across states, average rent is about **$1,366**, average density is **4.863 breweries per 100k**, average income is about **$79,769**, and average market potential is about **33,729 people per brewery**.

#### 5.3.4 Query 4 (join 3): Join-based “model-ready” extract (explicit join query)
- **What:** Pull a “model-ready” dataset directly via SQL joins (instead of using the prebuilt table), keeping only the columns used in modeling.
- **How:** Join the three base tables again, select modeling columns, and compute the same ratio features.
- **Why:** Demonstrates SQL join competency and provides an auditable, end-to-end SQL pipeline for the modeling dataset.

In [ ]:
model_ready_sql = pd.read_sql("""
SELECT
    b.state,
    b.region,
    c.total_population,
    c.weighted_avg_income,
    r.mean_rent,
    CAST(b.total_breweries AS FLOAT) / c.total_population * 100000.0 AS breweries_per_100k
FROM brewery_agg b
JOIN census_agg  c ON b.state = c.state
JOIN zillow_rent r ON b.state = r.state
ORDER BY breweries_per_100k ASC;
""", conn)

model_ready_sql.head(10)

,state,region,total_population,weighted_avg_income,mean_rent,breweries_per_100k
0,Mississippi,Southeast,1654737,54917.474151,1279.838792,1.027354
1,Oklahoma,Southwest,3210756,64852.758710,1071.541120,1.370394
2,Utah,West,3260704,98563.180535,1601.923826,1.380070
3,Louisiana,Southeast,2979355,58729.378342,1100.933409,1.476830
4,Texas,Southwest,23279221,79958.828372,1252.178632,1.512078
5,Alabama,Southeast,3418068,67313.745276,1148.726965,1.521327
6,Nevada,West,3050404,78515.663123,1665.841029,1.671910
7,Florida,Southeast,17284872,77216.088454,1699.128667,1.805047
8,Arizona,Southwest,6900619,83511.734264,1465.823731,1.811432
9,Georgia,Southeast,5469171,75289.156684,1324.635174,1.828431


**SQL takeaway (Query 4):** A join-based extraction reproduces the key modeling features directly from the raw SQL tables, confirming that our model inputs (income, rent, population, brewery density) are traceable and consistent with the merged table approach.

#### 5.3.5 Query 5 (subquery 1): Low-cost markets (rent below national average)
- **What:** Identify states where mean rent is below the national mean (low-cost markets).
- **How:** Filter `state_metrics` using an uncorrelated subquery.
- **Why:** Lower rent can reduce fixed costs and is relevant for site selection.

In [ ]:
low_rent_states = pd.read_sql("""
SELECT
    state,
    region,
    ROUND(mean_rent, 2)          AS mean_rent,
    ROUND(breweries_per_100k, 4) AS breweries_per_100k
FROM state_metrics
WHERE mean_rent < (
                    SELECT AVG(mean_rent)
                    FROM state_metrics
                    )
ORDER BY mean_rent ASC;
""", conn)

print(f"States with below-average rent: {len(low_rent_states)}")
low_rent_states.head(10)

States with below-average rent: 29


,state,region,mean_rent,breweries_per_100k
0,Kansas,Midwest,920.22,1.8846
1,North Dakota,Midwest,965.32,4.1172
2,West Virginia,Southeast,970.50,5.1114
3,Ohio,Midwest,979.10,3.6284
4,Missouri,Midwest,983.15,3.2213
5,Indiana,Midwest,987.36,3.3292
6,Illinois,Midwest,987.56,2.2870
7,Iowa,Midwest,1002.91,3.4673
8,Wisconsin,Midwest,1036.11,5.0576
9,Arkansas,Southeast,1039.49,2.0961


**SQL takeaway (Query 5):** There are **29 states** with below-average rent. These states represent a large pool of lower-cost markets, which may reduce fixed operating costs for new brewery entry.

#### 5.3.6 Query 6 (subquery 2): “Opportunity states” (high income + low competition)
- **What:** Find states with above-average income but below-average brewery density (underserved yet strong demand).
- **How:** Use two uncorrelated subqueries as thresholds for income and density.
- **Why:** This directly answers the business question: where are attractive markets not yet saturated?

In [ ]:
opportunity_states = pd.read_sql("""
SELECT
    state,
    region,
    ROUND(breweries_per_100k, 4)   AS breweries_per_100k,
    ROUND(weighted_avg_income, 2)  AS weighted_avg_income,
    ROUND(mean_rent, 2)            AS mean_rent,
    ROUND(people_per_brewery, 0)   AS people_per_brewery
FROM state_metrics
WHERE breweries_per_100k < (
                            SELECT AVG(breweries_per_100k)
                            FROM state_metrics
                            )
    AND weighted_avg_income > (
                                SELECT AVG(weighted_avg_income)
                                FROM state_metrics
                                )
ORDER BY weighted_avg_income DESC;
""", conn)

print(f"High-income, low-density states (prime targets): {len(opportunity_states)}")
opportunity_states

High-income, low-density states (prime targets): 13


,state,region,breweries_per_100k,weighted_avg_income,mean_rent,people_per_brewery
0,Maryland,Southeast,3.0256,109285.05,1344.45,33051.0
1,Virginia,Southeast,4.2406,106075.46,1333.03,23582.0
2,New Jersey,Northeast,1.8767,104574.84,1768.99,53284.0
3,California,West,2.4498,104267.42,2040.86,40820.0
4,Hawaii,West,2.4131,103139.48,2575.27,41440.0
5,Massachusetts,Northeast,3.2595,102647.78,1861.74,30679.0
6,Utah,West,1.3801,98563.18,1601.92,72460.0
7,Minnesota,Midwest,3.8095,92692.42,1172.32,26250.0
8,New York,Northeast,2.5681,90920.96,1414.25,38939.0
9,Connecticut,Northeast,4.1738,89587.88,1656.26,23959.0


**SQL takeaway (Query 6):** We identify **13 “opportunity states”** that combine above-average income with below-average brewery density. These states match our investment thesis: strong purchasing power with less competitive saturation.

#### 5.3.7 Query 7 (group by 1): Regional summary of key metrics
- **What:** Summarize rent, density, income, and market potential by region.
- **How:** `GROUP BY` region with `AVG()` and `COUNT(*)`.
- **Why:** Regional differences matter; this also provides interpretable context for later within-region ranking.

In [ ]:
regional_summary = pd.read_sql("""
SELECT
    region,
    COUNT(*)                         AS state_count,
    ROUND(AVG(mean_rent), 2)          AS avg_rent,
    ROUND(AVG(breweries_per_100k), 4) AS avg_density,
    ROUND(AVG(weighted_avg_income),2) AS avg_income,
    ROUND(AVG(people_per_brewery),0)  AS avg_market_potential
FROM state_metrics
GROUP BY region
ORDER BY avg_density ASC;
""", conn)

regional_summary

,region,state_count,avg_rent,avg_density,avg_income,avg_market_potential
0,Southwest,4,1263.02,2.3113,73247.76,54071.0
1,Southeast,14,1271.62,2.9156,73949.35,44381.0
2,Midwest,12,1044.63,3.9744,75600.29,28422.0
3,West,11,1699.40,6.2732,89645.14,27380.0
4,Northeast,9,1581.80,8.4880,85206.41,22957.0


**SQL takeaway (Query 7):** Regional patterns are clear: the **Northeast** has the highest average brewery density (more competition), while the **West** has the highest average rent (higher fixed costs). The **Southwest** shows the highest average people-per-brewery, suggesting stronger “underserved demand” signals relative to other regions.

#### 5.3.8 Query 8 (group by 2): Rent tier breakdown within each region
- **What:** Classify states into rent tiers (Low/Medium/High) and count them within each region.
- **How:** Use `CASE WHEN ... THEN ... END` to define tiers, then `GROUP BY region, rent_tier`.
- **Why:** Helps interpret whether “opportunity states” cluster in certain regions or rent environments.

In [ ]:
rent_tier = pd.read_sql("""
SELECT
    region,
    CASE
        WHEN mean_rent < 1200 THEN 'Low (<$1200)'
        WHEN mean_rent < 1600 THEN 'Medium ($1200–$1600)'
        ELSE 'High (>$1600)'
    END AS rent_tier,
    COUNT(*) AS state_count
FROM state_metrics
GROUP BY region, rent_tier
ORDER BY region, rent_tier;
""", conn)

rent_tier

,region,rent_tier,state_count
0,Midwest,Low (<$1200),12
1,Northeast,High (>$1600),6
2,Northeast,Low (<$1200),1
3,Northeast,Medium ($1200–$1600),2
4,Southeast,High (>$1600),1
5,Southeast,Low (<$1200),5
6,Southeast,Medium ($1200–$1600),8
7,Southwest,Low (<$1200),1
8,Southwest,Medium ($1200–$1600),3
9,West,High (>$1600),5


**SQL takeaway (Query 8):** Rent environments differ by region. The **Midwest** is concentrated in the low-rent tier, the **Northeast** is concentrated in the high-rent tier, and the **Southeast/West** span medium-to-high tiers. This helps interpret whether “opportunity states” are driven by demand/competition rather than simply cheap rent.

#### 5.3.9 Query 9 (window function 1): Create `state_ranked` with within-region ranks + deviations
- **What:** Rank states within each region on key metrics, and compute deviations from the regional average.
- **How:** Use window functions: `AVG(...) OVER (PARTITION BY region)` and `RANK() OVER (...)`.
- **Why:** A state’s attractiveness should be judged relative to its region, not only nationally.

In [ ]:
cursor.executescript("""
DROP TABLE IF EXISTS state_ranked;

CREATE TABLE state_ranked AS
SELECT
    state,
    region,
    total_breweries,
    total_population,
    weighted_avg_income,
    mean_rent,
    breweries_per_100k,
    people_per_brewery,
    income_per_brewery,

    -- Deviations from regional average
    mean_rent         - AVG(mean_rent)         OVER (PARTITION BY region) AS rent_vs_region,
    breweries_per_100k- AVG(breweries_per_100k)OVER (PARTITION BY region) AS density_vs_region,
    people_per_brewery- AVG(people_per_brewery)OVER (PARTITION BY region) AS potential_vs_region,

    -- Within-region ranks (note: lower rent & lower density are "better" for cost/competition, while higher people_per_brewery is "better" for market potential)
    RANK() OVER (PARTITION BY region ORDER BY people_per_brewery DESC) AS region_potential_rank,
    RANK() OVER (PARTITION BY region ORDER BY mean_rent ASC)          AS region_cost_rank,
    RANK() OVER (PARTITION BY region ORDER BY breweries_per_100k ASC) AS region_competition_rank

FROM state_metrics
ORDER BY region, region_competition_rank;
""")
conn.commit()

pd.read_sql("SELECT * FROM state_ranked LIMIT 8;", conn)

,state,region,total_breweries,total_population,weighted_avg_income,mean_rent,breweries_per_100k,people_per_brewery,income_per_brewery,rent_vs_region,density_vs_region,potential_vs_region,region_potential_rank,region_cost_rank,region_competition_rank
0,Kansas,Midwest,47,2493872,75977.501668,920.222480,1.884620,53061.106383,1616.542589,-124.408576,-2.089757,24638.788407,1,1,1
1,Illinois,Midwest,257,11237385,88164.916081,987.556483,2.287009,43725.233463,343.054148,-57.074573,-1.687368,15302.915487,2,6,2
2,Missouri,Midwest,142,4408124,73528.367587,983.145887,3.221325,31043.126761,517.805406,-61.485169,-0.753052,2620.808785,3,4,3
3,Indiana,Midwest,162,4865964,70037.047656,987.357262,3.329248,30036.814815,432.327455,-57.273794,-0.645129,1614.496839,4,5,4
4,Iowa,Midwest,91,2624534,74043.606510,1002.909452,3.467282,28841.032967,813.666006,-41.721604,-0.507094,418.714991,5,7,5
5,Ohio,Midwest,303,8350890,71170.096007,979.102635,3.628356,27560.693069,234.884805,-65.528421,-0.346021,-861.624906,6,3,6
6,Minnesota,Midwest,183,4803718,92692.421624,1172.322822,3.809549,26249.825137,506.515965,127.691766,-0.164827,-2172.492839,7,11,7
7,Nebraska,Midwest,62,1591593,72002.520901,1184.955336,3.895468,25670.854839,1161.330982,140.324280,-0.078908,-2751.463137,8,12,8


**SQL takeaway (Query 9):** Window functions allow state performance to be evaluated relative to regional peers. The `state_ranked` table quantifies each state’s deviation from its regional averages and provides within-region ranks for cost (rent), competition (density), and market potential (people per brewery).

#### 5.3.10 Query 10 (window function 2): Top 3 “best targets” per region (ROW_NUMBER)
- **What:** Select the top 3 states per region by a composite “target score” using within-region normalization logic.
- **How:** Use `ROW_NUMBER() OVER (PARTITION BY region ORDER BY ...)` to pick top-k per region.
- **Why:** This produces a practical shortlist for decision-makers while ensuring regional diversity.

In [ ]:
top_targets_by_region = pd.read_sql("""
WITH scored AS (
    SELECT
        state,
        region,
        mean_rent,
        breweries_per_100k,
        people_per_brewery,        
        (region_cost_rank + region_competition_rank + region_potential_rank) AS target_score,   -- prefer low rent (cost), low density (competition), high people_per_brewery (potential)
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY (region_cost_rank + region_competition_rank + region_potential_rank) ASC
        ) AS rn
    FROM state_ranked
)
SELECT
    region,
    state,
    ROUND(mean_rent, 2)          AS mean_rent,
    ROUND(breweries_per_100k, 4) AS breweries_per_100k,
    ROUND(people_per_brewery, 0) AS people_per_brewery,
    target_score
FROM scored
WHERE rn <= 3
ORDER BY region, target_score ASC;
""", conn)

top_targets_by_region

,region,state,mean_rent,breweries_per_100k,people_per_brewery,target_score
0,Midwest,Kansas,920.22,1.8846,53061.0,3
1,Midwest,Illinois,987.56,2.2870,43725.0,10
2,Midwest,Missouri,983.15,3.2213,31043.0,10
3,Northeast,New York,1414.25,2.5681,38939.0,7
4,Northeast,New Jersey,1768.99,1.8767,53284.0,10
5,Northeast,Connecticut,1656.26,4.1738,23959.0,13
6,Southeast,Mississippi,1279.84,1.0274,97337.0,8
7,Southeast,Louisiana,1100.93,1.4768,67713.0,8
8,Southeast,Alabama,1148.73,1.5213,65732.0,11
9,Southwest,Oklahoma,1071.54,1.3704,72972.0,3


**SQL takeaway (Query 10):** A top-3 shortlist per region translates the SQL ranking logic into actionable recommendations. Using a composite score based on within-region ranks (low rent, low density, high people-per-brewery), we generate a diversified set of priority targets by region rather than relying on national rankings alone.

### 5.4 Pull Results Back to Python
We pull the SQL outputs into Python so they can be reused in the EDA and Regression sections.

In [ ]:
state_final = pd.read_sql("SELECT * FROM state_ranked;", conn)  # Although this is also an SQL query, it is not used for analysis. So there is no further explanation.
print(f'Pulled {len(state_final)} states from SQLite')
state_final.head()

Pulled 50 states from SQLite


,state,region,total_breweries,total_population,weighted_avg_income,mean_rent,breweries_per_100k,people_per_brewery,income_per_brewery,rent_vs_region,density_vs_region,potential_vs_region,region_potential_rank,region_cost_rank,region_competition_rank
0,Kansas,Midwest,47,2493872,75977.501668,920.222480,1.884620,53061.106383,1616.542589,-124.408576,-2.089757,24638.788407,1,1,1
1,Illinois,Midwest,257,11237385,88164.916081,987.556483,2.287009,43725.233463,343.054148,-57.074573,-1.687368,15302.915487,2,6,2
2,Missouri,Midwest,142,4408124,73528.367587,983.145887,3.221325,31043.126761,517.805406,-61.485169,-0.753052,2620.808785,3,4,3
3,Indiana,Midwest,162,4865964,70037.047656,987.357262,3.329248,30036.814815,432.327455,-57.273794,-0.645129,1614.496839,4,5,4
4,Iowa,Midwest,91,2624534,74043.606510,1002.909452,3.467282,28841.032967,813.666006,-41.721604,-0.507094,418.714991,5,7,5
